# Lab: Face Recognition
**Instructor:** Eng. Mohamed Yossri Elwaly

Two approaches to face recognition:
1. **DeepFace**: High-level 1-liner
2. **InsightFace**: Raw 512-D ArcFace vectors + Cosine Similarity math

---
## 1. DeepFace (1-Line Verification)

In [1]:
from deepface import DeepFace

# 1:1 Verification
result = DeepFace.verify(
    img1_path="faces/actor1_img1.jpg",
    img2_path="faces/actor1_img2.jpg",
    model_name="ArcFace"
)

print("Same Person?:", result["verified"])
print("Distance:    ", round(result["distance"], 4))
print("Threshold:   ", round(result["threshold"], 4))


Same Person?: True
Distance:     0.443
Threshold:    0.68


---
## 2. InsightFace (The Engineering Approach)
$$\text{Cosine Similarity} = \mathbf{u}_{norm} \cdot \mathbf{v}_{norm}$$

In [2]:
import cv2
import numpy as np
from insightface.app import FaceAnalysis

# 1. Initialize detector + ArcFace embedder
app = FaceAnalysis(name='buffalo_sc', providers=['CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\moham/.insightface\models\buffalo_sc\det_500m.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\moham/.insightface\models\buffalo_sc\w600k_mbf.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)


In [3]:
# 2. Detect and extract 512-D embeddings
img1 = cv2.imread("faces/actor1_img1.jpg")
img2 = cv2.imread("faces/actor1_img2.jpg")

face1 = app.get(img1)[0]
face2 = app.get(img2)[0]

# 3. Normalize vectors to unit length
emb1 = face1.embedding / np.linalg.norm(face1.embedding)
emb2 = face2.embedding / np.linalg.norm(face2.embedding)

# 4. Cosine Similarity (Dot product)
sim = np.dot(emb1, emb2)
print("Similarity Score:", round(sim, 4))
print("Match?:          ", sim >= 0.45)

Similarity Score: 0.6499
Match?:           True


---
## 3. Mini-Project: 1:N Face Search (Smart Door / Attendance)

In [4]:
# Register Actor 2 into our system
img_actor2 = cv2.imread("faces/actor2_img1.jpg")
face_actor2 = app.get(img_actor2)[0]
emb_actor2 = face_actor2.embedding / np.linalg.norm(face_actor2.embedding)

# Database of enrolled users
database = {
    "Actor 1": emb1,
    "Actor 2": emb_actor2
}

# New incoming probe face (actor1_img2)
probe = emb2

# Search database
for name, enrolled_emb in database.items():
    score = np.dot(probe, enrolled_emb)
    print(f"Comparing with {name:8s} -> Similarity: {score:.4f}")
    if score >= 0.45:
        print(f"  >> [ACCESS GRANTED] Welcome {name}! ({score*100:.1f}%)\n")

Comparing with Actor 1  -> Similarity: 0.6499
  >> [ACCESS GRANTED] Welcome Actor 1! (65.0%)

Comparing with Actor 2  -> Similarity: -0.0337
